# DeepAR（M5）

参考：Salinas et al., DeepAR: Probabilistic Forecasting with Autoregressive Recurrent Networks (arXiv:1704.04110)

全商品で1つのLSTMを学習する。LSTMは前日の販売数と曜日から翌日の販売数の分布（負の二項分布）のパラメータμ, αを出す。
予測は分布からサンプルした値を次の日の入力にして14日分回し、それを複数回やって平均する。

MODE = 'val' で検証、'submit' で提出ファイル作成。

## 設定

In [ ]:
MODE = 'submit'          # 'val': d_1913まで学習してd_1914〜1927で評価 / 'submit': 全期間で学習して提出
CTX, PRED = 56, 14    # 入力に使う日数, 予測日数
HIDDEN, LAYERS = 120, 3
N_ITER, BATCH, LR = 3000, 512, 5e-3
N_SAMPLES = 50
SEEDS = [0, 1, 2, 3, 4]   # この seed で1本ずつ学習して、予測を平均する（[0] にすると1本だけ）

In [2]:
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## データ
Z: 販売数 (30490, 1927)、X: 日ごとの特徴量 (1969, 特徴量数)

SNAPは州ごとに日が違うので、features() で商品の州に合わせて付ける。

In [3]:
train = pd.read_csv('trainset.csv')
cal = pd.read_csv('calendar.csv')
Z = train.iloc[:, 6:].values.astype(np.float32)
N = len(Z)
T = 1913 if MODE == 'val' else Z.shape[1]   # T日目の手前までで学習

def make_features(cal):
    events = cal['event_type_1'].fillna('') + ',' + cal['event_type_2'].fillna('')
    feats = [
        pd.get_dummies(cal['weekday']).values,                            # 曜日 7列
        pd.get_dummies(cal['month']).values,                              # 月 12列
        pd.to_datetime(cal['date']).dt.day.values[:, None] / 31,          # 日付 1列
        np.stack([events.str.contains(e) for e in ['Sporting', 'Cultural', 'National', 'Religious']], 1),  # イベントの種類 4列
    ]
    return np.concatenate(feats, axis=1).astype(np.float32)

X = make_features(cal)
X = (X - X.mean(0)) / X.std(0)                 # 標準化：各列を平均0・分散1にそろえる
X = torch.tensor(X, device=device)
SNAP = torch.tensor(cal[['snap_CA', 'snap_TX', 'snap_WI']].values, dtype=torch.float32, device=device)
SNAP = (SNAP - SNAP.mean(0)) / SNAP.std(0)     # SNAPも同じく標準化
STATE = torch.tensor(train['state_id'].map({'CA': 0, 'TX': 1, 'WI': 2}).values, device=device)
Z_t = torch.tensor(Z, device=device)

# 埋め込み用：商品・部門・店舗をそれぞれ 0,1,2,... の番号にする
CAT_COLS = ['item_id', 'dept_id', 'store_id']
CATS = torch.tensor(np.stack([train[c].astype('category').cat.codes.values for c in CAT_COLS], 1), dtype=torch.long, device=device)
N_CATS = [train[c].nunique() for c in CAT_COLS]

def features(i, t):
    # 商品i (B,)、日付t (B, 日数) → 特徴量 (B, 日数, 特徴量数)
    snap = SNAP[t, STATE[i][:, None]].unsqueeze(-1)
    return torch.cat([X[t], snap], dim=-1)

N_FEAT = X.shape[1] + 1
print(Z.shape, X.shape, N_FEAT)

(30490, 1927) torch.Size([1969, 24]) 25


## モデル
入力は前日の販売数/νと特徴量。ν = 1 + 直近の平均販売数。
出力のμにはνを掛け、αは√νで割って元のスケールに戻す。

In [4]:
class DeepAR(nn.Module):
    def __init__(self, n_feat, n_cats, emb_dims=(20, 4, 4)):
        super().__init__()
        # 商品20次元・部門4次元・店舗4次元の埋め込み（番号 → 学習するベクトル）
        self.embs = nn.ModuleList([nn.Embedding(n, d) for n, d in zip(n_cats, emb_dims)])
        self.lstm = nn.LSTM(1 + n_feat + sum(emb_dims), HIDDEN, LAYERS, batch_first=True)
        self.head = nn.Linear(HIDDEN, 2)   # μ, α

    def forward(self, z_prev, x, nu, i, state=None):
        e = torch.cat([emb(CATS[i, k]) for k, emb in enumerate(self.embs)], dim=-1)  # 商品iの埋め込み (B, 28)
        e = e.unsqueeze(1).expand(-1, x.shape[1], -1)                                # 毎日同じものを付ける
        inp = torch.cat([(z_prev / nu).unsqueeze(-1), x, e], dim=-1)
        h, state = self.lstm(inp, state)
        mu, alpha = (F.softplus(self.head(h)) + 1e-4).unbind(-1)
        return mu * nu, alpha / nu.sqrt(), state

def nb(mu, alpha):
    # 負の二項分布 平均mu, 分散mu + mu^2 * alpha
    r = 1 / alpha
    return torch.distributions.NegativeBinomial(total_count=r, probs=mu / (mu + r))

## 学習
商品と開始日をランダムに選んで CTX+PRED 日の窓を切り出し、負の対数尤度を最小化する。

In [5]:
def get_batch():
    i = torch.randint(0, N, (BATCH,), device=device)
    s = torch.randint(T - 2 * 365, T - CTX - PRED + 1, (BATCH,), device=device)  # 直近2年から
    t = s[:, None] + torch.arange(CTX + PRED, device=device)
    z = Z_t[i[:, None], t]
    z_prev = Z_t[i[:, None], t - 1]
    nu = 1 + z[:, :CTX].mean(1, keepdim=True)
    return z, z_prev, features(i, t), nu, i

def train_model(seed):
    torch.manual_seed(seed)
    model = DeepAR(N_FEAT, N_CATS).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for it in range(1, N_ITER + 1):
        z, z_prev, x, nu, i = get_batch()
        mu, alpha, _ = model(z_prev, x, nu, i)
        loss = -nb(mu, alpha).log_prob(z).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if it % 500 == 0:
            print('seed', seed, it, round(loss.item(), 4))
    return model

models = [train_model(seed) for seed in SEEDS]   # seed ごとに1本ずつ学習

seed 0 500 1.0534


seed 0 1000 1.0182


seed 0 1500 1.0096


seed 0 2000 1.0042


seed 0 2500 0.9563


seed 0 3000 1.0062


... (20 outputs left out)

seed 4 1500 0.9674


seed 4 2000 1.084


seed 4 2500 0.9892


seed 4 3000 0.9812


## 予測
直近CTX日を入れてLSTMの状態を作り、そこから14日分サンプリング。各日のμをN_SAMPLES本で平均する。

In [6]:
@torch.no_grad()
def forecast(model):
    i = torch.arange(N, device=device)
    nu = 1 + Z_t[:, T - CTX:T].mean(1, keepdim=True)
    t_ctx = torch.arange(T - CTX, T, device=device).expand(N, -1)
    _, _, state0 = model(Z_t[:, T - CTX - 1:T - 1], features(i, t_ctx), nu, i)
    total = torch.zeros(N, PRED, device=device)
    for _ in range(N_SAMPLES):
        state, z = state0, Z_t[:, T - 1:T]
        for k in range(PRED):
            t_k = torch.full((N, 1), T + k, device=device)
            mu, alpha, state = model(z, features(i, t_k), nu, i, state)
            z = nb(mu, alpha).sample()
            total[:, k] += mu[:, 0]
    return (total / N_SAMPLES).cpu().numpy()

preds = [forecast(m) for m in models]   # seed ごとの予測
pred = np.mean(preds, axis=0)           # 小数のまま平均する（四捨五入は評価・提出のとき）
print(pred[:3].round(2))

[[1.05 0.94 0.86 0.83 0.97 1.22 1.21 0.91 0.83 0.83 0.82 0.96 1.2  1.2 ]
 [0.1  0.12 0.14 0.14 0.17 0.21 0.2  0.15 0.14 0.15 0.15 0.18 0.22 0.21]
 [0.41 0.41 0.44 0.47 0.58 0.74 0.71 0.51 0.47 0.48 0.49 0.59 0.73 0.7 ]]


## 評価（val）
評価指標は売上金額で重み付けしたRMSSE

In [7]:
def score(pred):
    hist, true = Z[:, :T], Z[:, T:T + PRED]
    start = (hist > 0).argmax(1)
    scale = np.array([np.mean(np.diff(h[s:]) ** 2) + 1e-8 for h, s in zip(hist, start)])
    price = pd.read_csv('sell_prices.csv')
    wk = cal['wm_yr_wk'].values[T - 28:T]
    p = price[price.wm_yr_wk.isin(wk)].pivot_table(index=['store_id', 'item_id'], columns='wm_yr_wk', values='sell_price')
    p = p.reindex(pd.MultiIndex.from_frame(train[['store_id', 'item_id']]))[wk].fillna(0).values
    w = (hist[:, -28:] * p).sum(1); w = w / w.sum()
    return {f'{h}日': round(float((w * np.sqrt(((true[:, :h] - pred[:, :h]) ** 2).mean(1) / scale)).sum()), 4) for h in (1, 14)}

if MODE == 'val':
    print('DeepAR', score(pred))
    print('DeepAR（四捨五入）', score(np.round(pred)))

## 提出（submit）
四捨五入して整数にする。

In [8]:
if MODE == 'submit':
    sub = pd.read_csv('submit.csv')
    sub.iloc[:, 6:] = np.round(pred).astype(int)
    out = f'submit_new_avg{len(SEEDS)}.csv'   # seed の本数を名前に入れる（今の submit_new.csv を上書きしない）
    sub.to_csv(out, index=False)
    print('saved', out)

saved submit_new_avg5.csv
